In [2]:
import pandas as pd
import glob

# ดึงข้อมูลและจัดการโครงสร้างตารางปี 63-66
# อ่านไฟล์ Excel โดยข้ามหัวตาราง 2 บรรทัดแรกที่ไม่จำเป็น
df_years = pd.read_excel("2555 - 2566.xlsx", header=2)

# เติมชื่อจังหวัดที่แหว่งไป (เนื่องจากไฟล์ Excel มีการ Merge Cell ทำให้ตอนอ่านเข้ามาจะเป็นค่า NaN)
df_years['จังหวัด'] = df_years['จังหวัด'].ffill()

# กรองเอาเฉพาะข้อมูล "รวมรถทุกชนิด" (ตัดแยกรถเก๋ง รถกระบะ มอเตอร์ไซค์ ทิ้งไป)
df_total = df_years[df_years['ชนิดรถยนต์'] == 'รวม'].copy()

# ลบช่องว่าง (Whitespace) ที่อาจซ่อนอยู่ในชื่อคอลัมน์ เพื่อป้องกัน Error ตอนเรียกใช้
df_total.columns = df_total.columns.astype(str).str.strip()

# ข้อมูลดิบมีบรรทัดสรุปยอดรวมของแต่ละภาคปนมาด้วย เราต้องตัดออกให้เหลือแค่ 77 จังหวัด
exclude_words = ['ทั่วราชอาณาจักร', 'ภาคกลาง', 'ภาคเหนือ', 'ภาคใต้', 'ภาคตะวันออก', 'ภาคตะวันตก', 'ภาคตะวันออกเฉียงเหนือ']
df_extract = df_total[~df_total['จังหวัด'].isin(exclude_words)][['จังหวัด', '2563', '2564', '2565', '2566']].copy()

# ลบช่องว่างหน้าและหลังชื่อจังหวัด เพื่อเตรียมนำไป Join กับตารางอื่น
df_extract['จังหวัด'] = df_extract['จังหวัด'].str.strip()

# ค้นหาไฟล์ CSV ทั้งหมดที่มีคำว่า "ภาค" ในชื่อไฟล์
files_2567 = glob.glob("2567.xls - ภาค*.csv")
provinces_2567 = {} # สร้าง Dictionary เพื่อเก็บชื่อจังหวัดและยอดรถปี 67

for f in files_2567:
    df_reg = pd.read_csv(f, header=2)
    # ค้นหาบรรทัดที่มีคำว่า 'รวมทั้งสิ้น' เพื่อดึงยอดรวมรถทั้งหมดของภาคนั้น
    row_total = df_reg[df_reg.iloc[:, 0].astype(str).str.contains('รวมทั้งสิ้น', na=False)]

    if not row_total.empty:
        # วนลูปอ่านค่าทีละคอลัมน์ (ซึ่งเป็นชื่อจังหวัด)
        for col in df_reg.columns[2:]:
            val = row_total[col].values[0]
            if pd.notna(val):
                prov_name = col.strip()
                # Data Cleaning: แก้ไขคำผิดที่มาจากไฟล์ต้นฉบับของกรมการขนส่งฯ
                if prov_name == 'แม่ฮองสอน': prov_name = 'แม่ฮ่องสอน'
                provinces_2567[prov_name] = float(val)

# ข้อมูลกรุงเทพฯ ในปี 67 แยกอยู่ในอีกไฟล์ (ไม่ได้อยู่ในไฟล์รายภาค) จึงต้องดึงแยก
df_all = pd.read_excel("2567.xls", header=3)
row_total_bkk = df_all[df_all.iloc[:, 0].astype(str).str.contains('รวมทั้งสิ้น', na=False)]
provinces_2567['กรุงเทพมหานคร'] = float(row_total_bkk['กรุงเทพฯ'].values[0])

# นำยอดรถปี 67 ที่ดึงมาได้ (Dictionary) ไป Map ต่อท้ายตารางหลักตามชื่อจังหวัด
df_extract['2567'] = df_extract['จังหวัด'].map(provinces_2567)

# ปัญหา: ข้อมูลรถยนต์เป็นค่า "รายปี" แต่ข้อมูลฝุ่น PM2.5 เป็นค่า "รายเดือน"
# วิธีแก้: ทำการขยายข้อมูลรถยนต์ 1 ปี ให้ซ้ำกัน 12 เดือน เพื่อให้สามารถ Join กับตารางฝุ่นได้
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
results = []

for idx, row in df_extract.iterrows():
    prov = row['จังหวัด']
    # วนลูป 5 ปี พร้อมแปลงปี พ.ศ. ให้เป็น ค.ศ. เพื่อง่ายต่อการนำไปวิเคราะห์
    for year_th, year_en in zip(['2563', '2564', '2565', '2566', '2567'], [2020, 2021, 2022, 2023, 2024]):
        val = row[year_th]

        # วนลูป 12 เดือน นำค่ายอดรถมาใส่ซ้ำ (Broadcasting)
        for m in months:
            results.append({
                'Year': year_en,
                'Month': m,
                'Province': prov,
                'Total_Vehicles': float(val) if pd.notna(val) else None
            })

# แปลงผลลัพธ์กลับเป็น DataFrame
df_monthly = pd.DataFrame(results)

# บันทึกไฟล์ พร้อมตั้งค่า encoding เป็น utf-8-sig เพื่อให้อ่านภาษาไทยใน Excel ได้ไม่เพี้ยน
df_monthly.to_csv('Cleaned_Vehicles_Monthly.csv', index=False, encoding='utf-8-sig')

# ตรวจสอบความถูกต้อง (ต้องได้ 4,620 แถว มาจาก 77 จังหวัด x 5 ปี x 12 เดือน)
print(f"จำนวนทั้งหมด: {len(df_monthly)} แถว (77 จังหวัด x 5 ปี x 12 เดือน)")
print(df_monthly.head())

จำนวนทั้งหมด: 4620 แถว (77 จังหวัด x 5 ปี x 12 เดือน)
   Year Month       Province  Total_Vehicles
0  2020   Jan  กรุงเทพมหานคร      10971799.0
1  2020   Feb  กรุงเทพมหานคร      10971799.0
2  2020   Mar  กรุงเทพมหานคร      10971799.0
3  2020   Apr  กรุงเทพมหานคร      10971799.0
4  2020   May  กรุงเทพมหานคร      10971799.0
